# Teacher dynamics of Li$_3$OCl at 1000, 900, 800 and 700 K

Production runs of the teacher with the batched engine: 32 replicas per temperature, the centre-of-mass velocity removed at every step, hops counted relative to the framework, and hop paths (0.5 ps before and after every hop, a frame every 10 fs) for the leverage analysis.

Input: `step5_bundle.zip` with `batched_md.py` and `li3ocl_teacher.model` from this archive. Runtime: GPU. `HOURS` sets the wall time of a session; with `USE_DRIVE = True` checkpoints and results are written to Google Drive, and running all cells again resumes an interrupted session.

In [ ]:
!pip -q install mace-torch ase
import torch, time, json, os, numpy as np
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
HOURS = 6.0                                   # total wall-time budget for this session
TEMPS = [1000.0, 900.0, 800.0, 700.0]         # K; budget is split equally among the temperatures that are not finished yet
NS_TARGET = {1000.0: 0.15, 900.0: 0.15, 800.0: 0.15, 700.0: 0.15} # ns PER REPLICA (x32 replicas) at which a temperature counts as finished; a T4 does ~0.8 ns in total per hour
USE_DRIVE = True
OUT = '.'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/li3ocl_step5'; os.makedirs(OUT, exist_ok=True)
    except Exception as err: print('no Drive, working locally:', err)
if not os.path.exists('batched_md.py'):
    from google.colab import files
    up = files.upload()            # choose step5_bundle.zip
    os.system('unzip -o -q step5_bundle.zip')
print(OUT, sorted(os.listdir('.')))

In [ ]:
from ase import Atoms
from mace.calculators import MACECalculator
from batched_md import BatchedMD, HopCounter
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; A = 3.926; B = 32; CHUNK = 5000
unit = Atoms('ClOLi3', scaled_positions=[(0,0,0), (.5,.5,.5), (.5,.5,0), (.5,0,.5), (0,.5,.5)], cell=[A]*3, pbc=True)
s = unit.repeat((3,3,3)); li_all = [i for i, z in enumerate(s.get_chemical_symbols()) if z == 'Li']; SITES = s.positions[li_all].copy(); del s[li_all[0]]
sym = s.get_chemical_symbols(); LI = np.array([i for i, z in enumerate(sym) if z == 'Li']); FW = np.array([i for i, z in enumerate(sym) if z != 'Li'])
calc = MACECalculator(model_paths='li3ocl_teacher.model', device=DEV, default_dtype='float32'); model = calc.models[0]
def counter(eng): return HopCounter(eng, LI, SITES, window=50, framework_idx=FW, framework_ref=s.positions[FW].mean(0))
def save_state(eng, hc, path):
    torch.save(dict(x=eng.x.cpu(), v=eng.v.cpu(), step=eng.step_count, assign=hc.assign.cpu(), hops=hc.hops.cpu(), x0=hc.x0.cpu(), msd=hc.msd, events=hc.events, frames=hc.frames,
                    saved_to=hc.saved_to, rec_until=hc.rec_until, buf=hc.buf), path + '.tmp'); os.replace(path + '.tmp', path)
def write_result(T, eng, hc):
    np.savez_compressed(f'{OUT}/li3ocl_teacher_{int(T)}K.npz', msd=np.array(hc.msd), hops=hc.hops.cpu().numpy(), events=np.array(hc.events), frame_step=np.array([f[0] for f in hc.frames]),
                        frame_replica=np.array([f[1] for f in hc.frames]), frames=np.array([f[2] for f in hc.frames]) if hc.frames else np.zeros((0, len(s), 3), np.float32),
                        sites=SITES, numbers=s.numbers, cell=s.cell.lengths(), ns_per_replica=eng.step_count * 2e-6, T=T)

In [ ]:
t_start = time.time(); SUMMARY = {}
def done_ns(T):
    ck = f'{OUT}/ckpt_{int(T)}K.pt'
    return torch.load(ck, weights_only=False)['step'] * 2e-6 if os.path.exists(ck) else 0.0
for T in TEMPS:
    todo = [t for t in TEMPS if done_ns(t) < NS_TARGET[t] and TEMPS.index(t) >= TEMPS.index(T)]
    left = HOURS * 3600 - (time.time() - t_start)
    if T not in todo or left < 300: continue
    budget = left / len(todo); ck = f'{OUT}/ckpt_{int(T)}K.pt'; rng = np.random.default_rng(int(T))
    x0 = np.stack([s.positions + rng.normal(0, 0.05, s.positions.shape) for _ in range(B)])
    eng = BatchedMD(model, s.numbers, s.cell.lengths(), x0, T, device=DEV, seed=int(T))
    if os.path.exists(ck):
        st = torch.load(ck, weights_only=False); eng.x, eng.v, eng.step_count = st['x'].to(DEV), st['v'].to(DEV), st['step']; eng._build_neighbours(); eng.e, eng.f = eng.energy_forces()
        hc = counter(eng); hc.assign, hc.hops, hc.x0 = st['assign'].to(DEV), st['hops'].to(DEV), st['x0'].to(DEV)
        hc.msd, hc.events, hc.frames, hc.saved_to, hc.rec_until, hc.buf = st['msd'], st['events'], st['frames'], st['saved_to'], st['rec_until'], st['buf']
        print(f'{T:.0f} K: resumed at {eng.step_count * 2e-6:.3f} ns per replica')
    else:
        eng.run(2500); eng.step_count = 0; hc = counter(eng)                       # 5 ps equilibration, then start the bookkeeping
    t0 = time.time()
    while eng.step_count * 2e-6 < NS_TARGET[T] and time.time() - t0 < budget:
        eng.run(CHUNK, callback=hc, callback_every=5); save_state(eng, hc, ck)
        ns = eng.step_count * 2e-6; h = int(hc.hops.sum())
        print(f'{T:.0f} K: {ns:.3f} ns/replica, hops {h} ({h / (B * ns):.0f} per ns), T = {eng.temperature().mean():.0f} K, path frames {len(hc.frames)}, {(time.time() - t_start) / 60:.0f} min', flush=True)
    write_result(T, eng, hc); ns = eng.step_count * 2e-6; h = hc.hops.cpu().numpy()
    SUMMARY[int(T)] = dict(ns_per_replica=round(ns, 4), hops_total=int(h.sum()), hops_per_ns=round(float(h.sum() / (B * ns)), 1), hops_per_replica_var_over_mean=round(float(h.var() / max(h.mean(), 1e-9)), 2),
                           msd_end_A2=round(float(hc.msd[-1].mean()), 3), path_frames=len(hc.frames), T_mean=round(float(eng.temperature().mean()), 1))

In [ ]:
print('=========== SUMMARY ==========='); print(json.dumps(dict(gpu=torch.cuda.get_device_name(0) if DEV == 'cuda' else 'cpu', hours=round((time.time() - t_start) / 3600, 2), results=SUMMARY), indent=1)); print('==============================')
os.system(f'cd {OUT} && zip -q li3ocl_step5_results.zip li3ocl_teacher_*K.npz && cp li3ocl_step5_results.zip /content/ 2>/dev/null')
from google.colab import files; files.download('/content/li3ocl_step5_results.zip' if os.path.exists('/content/li3ocl_step5_results.zip') else f'{OUT}/li3ocl_step5_results.zip')